In [1]:
pip install numpy matplotlib open3d

  Using cached open3d-0.19.0-cp312-cp312-manylinux_2_31_x86_64.whl.metadata (4.3 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached addict-2.4.0-py3-none-any.whl.metadata (1.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 684.8 kB/s eta 0:00:00a 0:00:01
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 497.3 kB/s eta 0:00:00 0:00:01
  Using cached pyquaternion-0.9.9-py3-none-any.whl.metadata (1.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached janus-2.0.0-py3-none-any.whl.metadata (5.3 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 1.1 MB/s eta 0:00:00a 0:00:0

In [20]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
pcd = o3d.io.read_point_cloud("DATA/TLS_kitchen.ply")

In [10]:
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [4]:
plane_model, inliers = pcd.segment_plane(distance_threshold=0.01, ransac_n=3, num_iterations=1000)

In [5]:
inlier_cloud = pcd.select_by_index(inliers)
outlier_cloud = pcd.select_by_index(inliers, invert=True)

In [6]:
inlier_cloud.paint_uniform_color([1.0, 0, 0])
outlier_cloud.paint_uniform_color([0.6, 0.6, 0.6])

PointCloud with 380071 points.

In [11]:
o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud])

In [13]:
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=16), fast_normal_computation=True)

In [16]:
labels = np.array(pcd.cluster_dbscan(eps=0.05, min_points=10))

In [21]:
max_label = labels.max()
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_geometries([pcd])

In [22]:
segment_models={}
segments={}

max_plane_idx=20

In [25]:
rest=pcd
for i in range(max_plane_idx):
    colors = plt.get_cmap("tab20")(i)    

    segment_models[i], inliers = rest.segment_plane(distance_threshold=0.01,ransac_n=3,num_iterations=1000)

    segments[i]=rest.select_by_index(inliers)    
    segments[i].paint_uniform_color(list(colors[:3]))    

rest = rest.select_by_index(inliers, invert=True)    
print("pass",i,"/",max_plane_idx,"done.")

pass 19 / 20 done.


In [ ]:
o3d.visualization.draw_geometries([segments[i] for i in range(max_plane_idx)]+[rest])

In [27]:
labels = np.array(segments[i].cluster_dbscan(eps=d_threshold*10, min_points=10))

NameError: name 'd_threshold' is not defined